# Structured questions

`instructions` and `criteria` can be JSON, not only flat strings. Use that when a question has several parts, or when the supporting data is already JSON. A short question can stay a string. The earlier notebooks are mostly strings. This one shows the structured form on the same shop.


In [ ]:
import sys
from datetime import date
from pathlib import Path
import json
import re
import statistics

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from langchain_typesafe import Choice, Noul, NoulCriteria, Score
from jev_examples.settings import ask, ask_many, draft, jev_model, openai_ready, show, typesafe_ready
from jev_examples.sample_data import (
    corpus_docs,
    customers,
    emails,
    load_json,
    lookup_order,
    open_incidents,
    order,
    products,
    read_text,
    ticket,
    tickets,
)

print("Jev model:", jev_model())
print("Jev key set:", typesafe_ready())
print("OpenAI key set:", openai_ready())


## 66. One field spec, several question types

The invoice field objects are shared by a Noul, a Choice, and a Score in one call.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    invoice = load_json("invoices.json")
    fields = {
        "invoice_number": {"name": "invoice_number", "description": "The identifier printed on the invoice."},
        "customer_name": {"name": "customer_name", "description": "Who the invoice was issued to."},
        "amount_due": {"name": "amount_due", "unit": "USD", "description": "The total to pay, not the shipping line."},
    }
    extracted = invoice["canned_extract"]
    response = ask(
        {"source_text": invoice["text"]},
        {
            "invoice_number_ok": Noul(
                instructions={
                    "field": fields["invoice_number"],
                    "extracted_value": extracted["invoice_number"],
                    "question": "Does `extracted_value` match the field in `source_text`?",
                }
            ),
            "customer_name": Choice(
                instructions={"field": fields["customer_name"], "question": "Which option is the field in `source_text`?"},
                criteria={name: None for name in invoice["name_candidates"]},
            ),
            "amount_band": Score(
                instructions={"field": fields["amount_due"], "question": "How large is the field in `source_text`?"},
                criteria=["Under $100", "$100 to $1,000", "Over $1,000"],
            ),
        },
    )
    show(response)
    print("customer:", response.choices["customer_name"].choice)


**What you should see.** The invoice number should check out. The customer should be Acme Holdings. The amount band for $240 should be the middle level, not the shipping line's 'under $100'.


## 67. Tell each option what it is not for

Neighboring teams steal each other's tickets when the criteria are only positive. `not_for` draws the border.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "department": Choice(
            instructions={"question": "Which team should handle this message?", "focus": "The shopper's main request."},
            criteria={
                "billing": {"what": "Charges, invoices, refunds", "not_for": "Tracking or login", "examples": ["I was charged twice"]},
                "orders": {"what": "Status, delivery, returns of goods", "not_for": "A duplicate card charge", "examples": ["Where is my package?"]},
                "account": {"what": "Login and password", "not_for": "Charges or delivery", "examples": ["I cannot log in"]},
            },
        )
    }
    for ticket_id in ("T-104", "T-118", "T-220"):
        response = ask(ticket(ticket_id)["body"], questions)
        show(response)
        print(ticket_id, "->", response.choices["department"].choice)


**What you should see.** T-104 should be billing, T-118 orders, T-220 account.


## 68. Pass the subtree as the option value

At the top of the catalog, each option's value is the child tree, so the model can see what lives under a branch before it commits.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    tree = load_json("taxonomy.json")
    listing = "Insulated travel mug for coffee, 16oz, leakproof lid."
    response = ask(
        listing,
        {"child": Choice(instructions="Which category does this product belong under?", criteria=tree)},
    )
    show(response)
    print("department:", response.choices["child"].choice)


**What you should see.** The travel mug should be Home & Kitchen, not Sporting Goods. The drinkware detail is inside that branch's value.


## 69. Score levels with signals

Each level has a summary and a few things to look for. This is PR-19, the one that bundles unrelated edits.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    pull = [item for item in load_json("prs.json") if item["id"] == "PR-19"][0]
    response = ask(
        pull,
        {
            "scope": Score(
                instructions={"question": "How focused is this change on a single edit?", "note": "Count independent changes, not line count."},
                criteria=[
                    {"summary": "One change", "signals": ["A single fix", "Tests for that fix"]},
                    {"summary": "One main change plus a small tweak", "signals": ["A primary edit and one related cleanup"]},
                    {"summary": "Several unrelated changes", "signals": ["Auth plus CI", "No tests", "Could be separate pull requests"]},
                ],
            )
        },
    )
    show(response)
    print("level:", round(response.scores["scope"].score))


**What you should see.** PR-19 should sit toward the top of the scale: several unrelated changes.


## 70. Pin down a subtle yes and no

Asking someone to reply with a password is not the same as telling them to use the reset page.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "requests_credentials": Noul(
            instructions={
                "question": "Does the message ask the recipient to disclose a password or code?",
                "focus": "A request to send the secret itself, not a request to reset it.",
            },
            criteria=NoulCriteria(
                true={"what": "Asks the person to reply with a password or code", "examples": ["Reply with your password"]},
                false={"what": "No secret is requested", "not_for": "A normal reset instruction", "examples": ["Reset your password from the settings page"]},
            ),
        )
    }
    for message in load_json("messages.json")["phishing"]:
        response = ask({"message": message}, questions)
        show(response)
        print(message["subject"], round(response.nouls["requests_credentials"].noul, 2))


**What you should see.** The prize subject should be high. The reset-link subject should be low.


## 71. Point at a nested field with a path

When the state has several parts, name the part in backticks so questions do not bleed into each other.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    state = {
        "support": {"tickets": [{"message": ticket("T-104")["body"]}, {"message": ticket("T-220")["body"]}]},
        "commerce": {"orders": [{"id": "A-104", "charges": [{"amount_usd": 49}, {"amount_usd": 49}]}]},
        "account": {"password_reset": "Email a reset link to the address on file."},
    }
    response = ask(
        state,
        {
            "duplicate_charge": Noul(instructions="Do `support.tickets[0].message` and `commerce.orders[0].charges` indicate a duplicate charge?"),
            "reset_supported": Noul(instructions="Can `account.password_reset` resolve `support.tickets[1].message`?"),
        },
    )
    show(response)


**What you should see.** Both probabilities should be high: the first ticket matches two $49 charges, and the reset instruction matches the login ticket.


## 72. Pass a schema fragment as JSON

Do not flatten a parameter spec into a sentence. Hand the fragment over and ask if the provided value fits.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    schema = load_json("tools.json")["schema"]
    good = {"order_id": "A-118"}
    bad = {"order_id": 118}
    for args in (good, bad):
        questions = {
            "valid_%s" % name: Noul(
                instructions={
                    "parameter": {"name": name, **spec},
                    "provided_value": args.get(name),
                    "question": "Is `provided_value` a valid value for `parameter`, given `user_request`?",
                }
            )
            for name, spec in schema["properties"].items()
        }
        response = ask({"user_request": "Where is order A-118?"}, questions)
        show(response)
        print(args, "->", "ok" if response.nouls["valid_order_id"].noul > 0.6 else "fix")


**What you should see.** `A-118` as a string should be ok. The bare number 118 should not, because the parameter is a string order id.
